# Step 2. Train Keypoint R-CNN for vertebra detection

Trains `torchvision` Keypoint R-CNN (ResNet-50 FPN backbone pretrained on ImageNet) to detect each lumbar vertebra and its four corner points. Anchors use aspect ratios from 0.25 to 4 to fit vertebral bodies. Training runs 10 epochs, then 50 more from the saved weights (SGD, learning rate 0.001, step decay), with COCO evaluation on the validation split after each epoch.

**Input:** dataset folder with `images/{train,val}` and `labels/{train,val}` (JSON from step 1). **Output:** model weights (`.pth`).

Training helpers (`engine`, `utils`, `transforms`) come from the torchvision detection references, through the [scoliovis-training](https://github.com/Blankeos/scoliovis-training) repository, with `kpt_oks_sigmas` adapted to four vertebral keypoints.

Code comments are in Spanish.

In [ ]:
# Montar el Drive de Google
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Establecer los parametros
dataset_dir = '/CARPETA_BASE_DE_DATOS' #Colocar ruta a base de datos en formato adecuado
model_save_path = 'MODELO.pth' #Colocar ruta deseada para guardar el modelo generado

# 1. Elementos Importados

In [ ]:
!pip install pycocotools &> /dev/null # no outputs `&> /dev/null`

In [ ]:
import os, json, cv2, numpy as np, matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import functional as F

# Importar biblioteca para aumentaciones
import albumentations as A

In [ ]:
!git clone https://github.com/Blankeos/scoliovis-training.git #Cloner repositorio de GitHub de Scoliovis
%cd scoliovis-training

In [ ]:
# Fue obtenido aqui: https://github.com/Blankeos/scoliovis-training.git
import transforms, utils, engine, train
from utils import collate_fn
from engine import train_one_epoch, evaluate

# Originalmente obtenido de aqui pero se modificaron `kpt_oks_sigmas` para adaptarse a nuestra base de datos
# https://github.com/pytorch/vision/tree/main/references/detection o https://github.com/alexppppp/keypoint_rcnn_training_pytorch.git

# 2. Aumentaciones

In [ ]:
 def train_transform():
     return A.Compose([
         A.Sequential([
             A.RandomRotate90(p=1), # Rotacion aleatoria de una imagen por 90 grados un total de 0 o mas veces
             A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, brightness_by_max=True, always_apply=False, p=1), # Cambios aleatorios de brillo y contraste
         ], p=1)
     ],
     keypoint_params=A.KeypointParams(format='xy'), # Mas acerca de los formatos de puntos clave usados. Leer mas https://albumentations.ai/docs/getting_started/keypoints_augmentation/
     bbox_params=A.BboxParams(format='pascal_voc', label_fields=['bboxes_labels']) # Los cuadros delimitadores deben tener rotulos. Leer mas: https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/
     )

# 3. Clase de Base de Datos

In [ ]:
class ClassDataset(Dataset):
    def __init__(self, root, split="val"):
        """
        split can be "train" | "val"
        """
        self.root = root
        self.split = split
        self.imgs_files = sorted(os.listdir(os.path.join(root, "images", split))) # string[]
        self.annotations_files = sorted(os.listdir(os.path.join(root, "labels", split))) # string[]



    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.split, self.imgs_files[idx])
        annotations_path = os.path.join(self.root, "labels", self.split, self.annotations_files[idx])

        img_original = cv2.imread(img_path)
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)

        with open(annotations_path) as f:
            data = json.load(f)
            bboxes_original = data['boxes']
            keypoints_original = data['keypoints']
            labels_original = data['labels']

            # Todos los objetos son anclas
            bboxes_labels_original = ['vert' for _ in bboxes_original]

        img, bboxes, keypoints = img_original, bboxes_original, keypoints_original

        # Convertir todo en un tensor de PyTorch
        bboxes = torch.as_tensor(bboxes, dtype=torch.float32)

        target = {}
        target["boxes"] = bboxes
        target["labels"] = torch.as_tensor([1 for _ in bboxes], dtype=torch.int64) # Todos los objetos son anclas
        target["image_id"] = torch.tensor([idx])
        target["area"] = (bboxes[:, 3] - bboxes[:, 1]) * (bboxes[:, 2] - bboxes[:, 0])
        target["iscrowd"] = torch.zeros(len(bboxes), dtype=torch.int64)
        target["keypoints"] = torch.as_tensor(keypoints, dtype=torch.float32)
        img = F.to_tensor(img)

        bboxes_original = torch.as_tensor(bboxes_original, dtype=torch.float32)
        target_original = {}
        target_original["boxes"] = bboxes_original
        target_original["labels"] = torch.as_tensor([1 for _ in bboxes_original], dtype=torch.int64) # Todos los objetos son anclas
        target_original["image_id"] = torch.tensor([idx])
        target_original["area"] = (bboxes_original[:, 3] - bboxes_original[:, 1]) * (bboxes_original[:, 2] - bboxes_original[:, 0])
        target_original["iscrowd"] = torch.zeros(len(bboxes_original), dtype=torch.int64)
        target_original["keypoints"] = torch.as_tensor(keypoints_original, dtype=torch.float32)
        img_original = F.to_tensor(img_original)

        return img, target

    def __len__(self):
        return len(self.imgs_files)

# 4. Visualizar un Elemento Aleatorio de la Base de Datos

In [ ]:
dataset = ClassDataset(dataset_dir, split="val")
data_loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

iterator = iter(data_loader)
batch = next(iterator)

print(f"Image Tensor (X):", batch[0])
print(f"Targets Dict[Tensor] (y):", batch[1])

In [ ]:
keypoints_classes_ids2names = {0: 'p1', 1: 'p2', 2: 'p3', 3: 'p4'}

def visualize(image, bboxes, keypoints=None, image_original=None, bboxes_original=None, keypoints_original=None, scores=None, keypoints_scores=None, fontScale=1):
    fontsize = 18

    for idx, bbox in enumerate(bboxes):
        start_point = (bbox[0], bbox[1])
        end_point = (bbox[2], bbox[3])
        image = cv2.rectangle(image.copy(), start_point, end_point, (0,255,0), 2)
        if (scores != None):
          image = cv2.putText(image.copy(), f"{idx+1} vert: {scores[idx]:.2f}", start_point, cv2.FONT_HERSHEY_SIMPLEX, fontScale, (0,255,0), 2, cv2.LINE_AA)

    if (keypoints != None):
      for instance_idx, kps in enumerate(keypoints):
          for idx, kp in enumerate(kps):
              image = cv2.circle(image.copy(), tuple(kp), 2, (255,0,0), 10)
              if (keypoints_scores != None):
                image = cv2.putText(image.copy(), f" {keypoints_classes_ids2names[idx]}: {keypoints_scores[instance_idx][idx]:.2f}", tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, fontScale, (255,0,0), 1, cv2.LINE_AA)
              else:
                image = cv2.putText(image.copy(), f" {keypoints_classes_ids2names[idx]}", tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, fontScale, (255,0,0), 2, cv2.LINE_AA)

    if image_original is None and keypoints_original is None:
        plt.figure(figsize=(40,40))
        plt.imshow(image)
        plt.axis("off")
    else:
        for bbox in bboxes_original:
            start_point = (bbox[0], bbox[1])
            end_point = (bbox[2], bbox[3])
            image_original = cv2.rectangle(image_original.copy(), start_point, end_point, (0,255,0), 2)

        for kps in keypoints_original:
            for idx, kp in enumerate(kps):
                image_original = cv2.circle(image_original, tuple(kp), 5, (255,0,0), 10)
                image_original = cv2.putText(image_original, " " + keypoints_classes_ids2names[idx], tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1, cv2.LINE_AA)

        f, ax = plt.subplots(1, 2, figsize=(40, 20))

        ax[0].imshow(image_original)
        ax[0].set_title('Original image', fontsize=fontsize)

        ax[1].imshow(image)
        ax[1].set_title('Transformed image', fontsize=fontsize)

image = (batch[0][0].permute(1,2,0).numpy() * 255).astype(np.uint8)
bboxes = batch[1][0]['boxes'].detach().cpu().numpy().astype(np.int32).tolist()

keypoints = []
for kps in batch[1][0]['keypoints'].detach().cpu().numpy().astype(np.int32).tolist():
    keypoints.append([kp[:2] for kp in kps])

visualize(image, bboxes, keypoints)

# 5. Entrenamiento

In [ ]:
def get_model(num_keypoints, weights_path=None):

    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   pretrained_backbone=True,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes = 2, # El fondo es la primera clase, el objeto es la segunda clase.
                                                                   rpn_anchor_generator=anchor_generator)
    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)

    return model

In [ ]:
# Cargar el Modelo y la Base de Datos
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

dataset_train = ClassDataset(dataset_dir, "train")
dataset_test = ClassDataset(dataset_dir, "val")

data_loader_train = DataLoader(dataset_train, batch_size=3, shuffle=True, collate_fn=collate_fn)
data_loader_test = DataLoader(dataset_test, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = get_model(num_keypoints = 4)
model.to(device)
None

In [ ]:
# Ciclo de Entrenamiento
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)
num_epochs = 10

for epoch in range(num_epochs):
    train_one_epoch(model, optimizer, data_loader_train, device, epoch, print_freq=1000)
    lr_scheduler.step()
    evaluate(model, data_loader_test, device)

# Guardar el modelo posterior al entrenamiento
torch.save(model.state_dict(), model_save_path)

In [ ]:
from engine import evaluate
evaluate(model, data_loader_test, device)

# 6. Visualizar Predicciones del Modelo

In [ ]:
# Crear Iterador
iterator = iter(data_loader_test)

In [ ]:
# Obtener Lote
images, targets = next(iterator)

# Establecer el tensor de la imagen a el dispositivo correcto y convertir a lista
images = list(image.to(device) for image in images)

# Inferencia
with torch.no_grad():
    model.to(device) # 1. Establecer a dispositivo
    model.eval() # 2. Establecer el modelo en modo 'evaluar'
    output = model(images) # 3. Obtener salida, las imagenes deben ser una lista de 3 tensores en forma [C, H, W]

print("Predictions: \n", output)

In [ ]:
output[0]['scores'].detach().cpu().numpy()

In [ ]:
image = (images[0].permute(1,2,0).detach().cpu().numpy() * 255).astype(np.uint8)
scores = output[0]['scores'].detach().cpu().numpy()

high_scores_idxs = np.where(scores > 0.7)[0].tolist() # Indices de cuadros con puntajes > 0.7
post_nms_idxs = torchvision.ops.nms(output[0]['boxes'][high_scores_idxs], output[0]['scores'][high_scores_idxs], 0.3).cpu().numpy() # Indices de cuadros restantes despues de aplicar NMS (umbral de iou=0.3)

# Abajo, en output[0]['keypoints'][high_scores_idxs][post_nms_idxs] and output[0]['boxes'][high_scores_idxs][post_nms_idxs]
# Primero, escogemos unicamente esos objetos, los cuales tienen puntajes por arriba del umbral definido. Esto se logra con elementos de seleccion con indices [high_scores_idxs]
# Segundo, escogemos unicamente esos objetos, los restantes cuando el NMS es aplicado. Esto se logra con elementos de seleccion con indices [post_nms_idxs] indeces

keypoints = []
for kps in output[0]['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    keypoints.append([list(map(int, kp[:2])) for kp in kps])

bboxes = []
for bbox in output[0]['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    bboxes.append(list(map(int, bbox.tolist())))

keypoints_scores = []
for kp_scores in output[0]['keypoints_scores'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    keypoints_scores.append(list(kp_scores.tolist()))

scores = scores.tolist()
# for kp_scores in output[0]['scores'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
#     scores.append(list(kp_scores.tolist()))

visualize(image, bboxes, keypoints, scores=scores)

# 7. Inferencia
Creando una funcion de inferencia modular para que sea mas sencillo reutilizarla.

In [ ]:
import torch
import torchvision
from torchvision.transforms import functional as F
import numpy
from torchvision.models.detection.rpn import AnchorGenerator

def get_kprcnn_model(num_keypoints, weights_path=None):
    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   pretrained_backbone=True,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes = 2, # El fondo es la primera clase, el objeto es la segunda clase.
                                                                   rpn_anchor_generator=anchor_generator)

    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)

    return model

def _open_image_path(path):
  img = cv2.imread(path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  img = F.to_tensor(img)
  return img

device = device=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model(num_keypoints = 4, weights_path='MODELO.pth') #Colocar ruta de modelo generado
model.to(device)

def _filter_output(output):
  # 1. Obtener puntajes
  scores = output['scores'].detach().cpu().numpy()

  # 2. Obtener los indices de los puntajes por encima del umbral
  high_scores_idxs = np.where(scores > 0.5)[0].tolist() # Indices de los puntajes > 0.7

  # 3. Obtener indices despues de la supresion de no-maximos
  post_nms_idxs = torchvision.ops.nms(output['boxes'][high_scores_idxs], output['scores'][high_scores_idxs], 0.3).cpu().numpy() # Indexes of boxes left after applying NMS (iou_threshold=0.3)

  # 4. Obtener 'bboxes` , `keypoints` y `scores` finales basados en los indices
  np_keypoints = output['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy()
  np_bboxes = output['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy()
  np_scores = output['scores'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy()

  # 5. Obtener los 17 puntajes mas altos
  sorted_scores_idxs = np.argsort(-1*np_scores) # descenso

  np_scores = scores[sorted_scores_idxs][:17]
  np_keypoints = np.array([np_keypoints[idx] for idx in sorted_scores_idxs])[:17]
  np_bboxes = np.array([np_bboxes[idx] for idx in sorted_scores_idxs])[:17]

  # 6. Sortear por medio de ymin
  # kp[0] es el primer punto en [p1,p2,p3,p4]
  # kp[0][1] es el y1 en p1=[x1,y1,x2,y2]
  ymins = np.array([kps[0][1] for kps in np_keypoints])

  sorted_ymin_idxs = np.argsort(ymins) # ascenso

  np_scores = np.array([np_scores[idx] for idx in sorted_ymin_idxs])
  np_keypoints = np.array([np_keypoints[idx] for idx in sorted_ymin_idxs])
  np_bboxes = np.array([np_bboxes[idx] for idx in sorted_ymin_idxs])

  # 7. Convertir todo en una lista envez de Numpy
  keypoints_list = []
  for kps in np_keypoints:
      keypoints_list.append([list(map(int, kp[:2])) for kp in kps])

  bboxes_list = []
  for bbox in np_bboxes:
      bboxes_list.append(list(map(int, bbox.tolist())))

  scores_list = np_scores.tolist()

  return bboxes_list, keypoints_list, scores_list

def predict(images):
  """
  images:
    > List of Tensors, shape=[C, W, H]. Values 0-1. |
    > Numpy array of image |
    > String path to image |
    > List of String paths to images

    returns (bboxes, keypoints, scores)[] of n=17
  """
  # 1. Procesar `images`
  images_input = [] # Lista de Tensores, forma=[C, W, H]. Valores 0-1.

  # 1.1 Para [Ruta string a imagen] & [Lista de rutas string a imagenes]
  if (type(images) == str):
    images_input.append(_open_image_path(images))
  elif isinstance(images, list):
    if (type(images[0]) == str):
      images_input = [_open_image_path(path) for path in images]
  elif torch.is_tensor(images):
    images_input = images
  elif isinstance(images, np.ndarray):
    print("A numpy array")
    images_input = [F.to_tensor(images)]

  images_input = [image.to(device) for image in images_input]

  # 2. Inferencia
  with torch.no_grad():
    model.to(device) # 1. Establecer en dispositivo
    model.eval() # 2. Establecer en modo evaluacion
    outputs = model(images_input) # 3. Obtener la salida

  filtered_outputs = [_filter_output(output) for output in outputs]

  return filtered_outputs

In [ ]:
image_paths = ['RUTA_IMAGEN_EJEMPLO'] #Colocar ruta de imagen para inferencia
image1 = cv2.imread(image_paths[0])
image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)

results = predict(image1)

In [ ]:
bboxes, keypoints, scores = results[0]
len(scores)

In [ ]:
image = cv2.cvtColor(cv2.imread(image_paths[0]), cv2.COLOR_BGR2RGB)

visualize(image, bboxes=bboxes, scores=scores)

In [ ]:
def kprcnn_to_scoliovis_api_format(bboxes, keypoints, scores):
  """
  detections: Array(17)
  { class: number, confidence: number, name: "vert", xmax: number, xmin: number, ymin: number, ymax: number }
  normalized_detections: Array(17)
  landmarks: Array(136)
  """
  detections = []
  for idx, bbox in enumerate(bboxes):
    detections.append({
        "class": 0,
        "confidence": scores[idx],
        "name": "vert",
        "xmin": bbox[0],
        "ymin": bbox[1],
        "xmax": bbox[2],
        "ymax": bbox[3],
    })
  landmarks = []
  for kps in keypoints:
    for kp in kps:
      landmarks.append(kp[0])
      landmarks.append(kp[1])

  return {
      "detections": detections,
      "landmarks": landmarks
  }

In [ ]:
bboxes, keypoints, scores = results[0]
kprcnn_to_scoliovis_api_format(bboxes, keypoints, scores)

## 8. Re-entrenamiento

In [ ]:
# Cargar el Modelo y la Base de Datos
MODEL_PATH = 'MODELO.pth' #Coloca ruta de modelo generado
RETRAINED_MODEL_SAVE_PATH = 'MODELO_V2.pth' #Coloca la ruta del modelo que será guardado al reentrenar.

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

dataset_train = ClassDataset(dataset_dir, "train")
dataset_test = ClassDataset(dataset_dir, "val")

data_loader_train = DataLoader(dataset_train, batch_size=3, shuffle=True, collate_fn=collate_fn)
data_loader_test = DataLoader(dataset_test, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = get_model(num_keypoints = 4, weights_path=MODEL_PATH)
model.to(device)
None

In [ ]:
# Ciclo de Entrenamiento
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)
num_epochs = 50

for epoch in range(num_epochs):
    train_one_epoch(model, optimizer, data_loader_train, device, epoch, print_freq=1000)
    lr_scheduler.step()
    evaluate(model, data_loader_test, device)

# Guardar el modelo posterior a re-entrenamiento
torch.save(model.state_dict(), RETRAINED_MODEL_SAVE_PATH)

In [ ]:
evaluate(model, data_loader_test, device)

IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.758
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.982
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.908
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.758
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.776
IoU metric: keypoints
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.687
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.940
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.775
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.687
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.703